# CNN — GAN augmentation comparison (per seed / scene / ratio / GAN)
Reads pre-generated `train_{gan}_augmented_ratio{ratio}.csv`, evaluates on the fixed `eval/` val/test, and compares every GAN. Resumable: re-running skips rows already in `cnn_gan_compare_raw.csv`.

In [ ]:
import os
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import sys; sys.path.insert(0, "../")
from util import *
from models.cnn import CNN

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

# ---- experiment grid ---------------------------------------------------
BASE     = "../CSV_Files"
EVAL_DIR = "../CSV_Files/eval"          # fixed val/test, never changes
GANS     = ["dcgan", "cgan", "wgan", "timegans"] # add "timegans" when generated
SEEDS    = [1, 2, 3, 4, 5]
SCENES   = range(1, 7)
RATIOS   = [0.0, 0.5, 1.0, 2.0]
EPOCHS, LR, WD = 70, 1e-2, 1e-4

# augmented training file for one (seed, scene, gan, ratio)
def train_path(seed, scene, gan, ratio):
    return f"{BASE}/seed{seed}/scene{scene}/train_{gan}_augmented_ratio{ratio}.csv"

RESULTS_CSV = "cnn_gan_compare_raw.csv"


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:, -1].astype("int64"))
    return X, y

def tensors_from_arrays(train_arr, val_arr, test_arr):
    X_tr_n, X_va_n, X_te_n, _ = preprocess_scenario(train_arr, val_arr, test_arr)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)

def make_model(device, seed=0):
    torch.manual_seed(seed)
    model = CNN(n_classes=8).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    return model

def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    tl = DataLoader(TensorDataset(X_tr[perm], y_tr[perm]), batch_size=batch_size, shuffle=False)
    vl = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    te = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return tl, vl, te

def train_one_epoch(model, loader, crit, opt, device):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval(); yt, yp = [], []
    for xb, yb in loader:
        yp.append(model(xb.to(device)).argmax(1).cpu().numpy()); yt.append(yb.numpy())
    return np.concatenate(yt), np.concatenate(yp)

def run_from_arrays(train_arr, val_arr, test_arr, device, seed=0):
    (X_tr,y_tr),(X_va,y_va),(X_te,y_te) = tensors_from_arrays(train_arr, val_arr, test_arr)
    tl, vl, te = make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, 50, seed)
    model = make_model(device, seed=seed)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    sch = optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
    for _ in range(EPOCHS):
        train_one_epoch(model, tl, crit, opt, device); sch.step()
    yt, yp = predict_all(model, te, device)
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(yt, yp), "precision": p, "recall": r,
            "f1": f, "n_train": len(X_tr)}


In [3]:
# fixed evaluation set (loaded once)
val_arr  = pd.read_csv(f"{EVAL_DIR}/val.csv").to_numpy()
test_arr = pd.read_csv(f"{EVAL_DIR}/test.csv").to_numpy()

# resume support: skip (gan,seed,scene,ratio) already in the results file
done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev.gan, prev.seed.astype(int), prev.scene.astype(int), prev.ratio.astype(float)))

cols = ["gan","seed","scene","ratio","accuracy","precision","recall","f1","n_train"]
for gan in GANS:
    for seed in SEEDS:
        for scene in SCENES:
            for ratio in RATIOS:
                key = (gan, seed, scene, float(ratio))
                if key in done:
                    continue
                path = train_path(seed, scene, gan, ratio)
                if not os.path.exists(path):
                    print(f"MISSING {path}"); continue
                train_arr = pd.read_csv(path).to_numpy()
                res = run_from_arrays(train_arr, val_arr, test_arr, device, seed=seed)
                row = {"gan": gan, "seed": seed, "scene": scene, "ratio": ratio, **res}
                pd.DataFrame([row])[cols].to_csv(
                    RESULTS_CSV, mode="a", header=not os.path.exists(RESULTS_CSV), index=False)
                done.add(key)
                print(f"{gan} seed{seed} scene{scene} ratio{ratio}: F1={res['f1']:.4f}")
print("done")


dcgan seed1 scene2 ratio1.0: F1=0.8576


dcgan seed1 scene2 ratio2.0: F1=0.8414


dcgan seed1 scene3 ratio0.0: F1=0.6897


dcgan seed1 scene3 ratio0.5: F1=0.6842


dcgan seed1 scene3 ratio1.0: F1=0.7060


...

wgan seed5 scene5 ratio2.0: F1=0.5908


wgan seed5 scene6 ratio0.0: F1=0.3533


wgan seed5 scene6 ratio0.5: F1=0.3785


wgan seed5 scene6 ratio1.0: F1=0.3833


wgan seed5 scene6 ratio2.0: F1=0.4242
done


In [4]:
df = pd.read_csv(RESULTS_CSV)

# mean / std over seeds, per (gan, scene, ratio)
agg = df.groupby(["gan","scene","ratio"])["f1"].agg(["mean","std"]).reset_index()
agg.to_csv("cnn_gan_compare_agg.csv", index=False)

# delta vs the no-augmentation baseline (ratio 0.0) within each (gan, scene)
base = (agg[agg.ratio==0.0][["gan","scene","mean"]]
        .rename(columns={"mean":"base"}))
cmp = agg.merge(base, on=["gan","scene"])
cmp["delta_vs_base"] = cmp["mean"] - cmp["base"]
cmp.to_csv("cnn_gan_compare_delta.csv", index=False)

# readable pivot: best augmented F1 per gan x scene (over ratios>0)
aug = cmp[cmp.ratio>0]
best = (aug.loc[aug.groupby(["gan","scene"])["mean"].idxmax()]
           [["gan","scene","ratio","mean","delta_vs_base"]]
           .rename(columns={"ratio":"best_ratio","mean":"best_f1"}))
print("Best augmented F1 per GAN x scene (vs baseline):")
print(best.to_string(index=False))
print("\nFull mean+/-std table saved to cnn_gan_compare_agg.csv")


Best augmented F1 per GAN x scene (vs baseline):
  gan  scene  best_ratio  best_f1  delta_vs_base
 cgan      1         0.5 0.892387      -0.002809
 cgan      2         0.5 0.865902       0.011638
 cgan      3         0.5 0.695189       0.004375
 cgan      4         2.0 0.649498       0.001087
 cgan      5         2.0 0.639519       0.028913
 cgan      6         2.0 0.409559       0.062213
dcgan      1         0.5 0.887158      -0.007995
dcgan      2         0.5 0.856048       0.002295
dcgan      3         2.0 0.711144       0.020498
dcgan      4         2.0 0.667201       0.018042
dcgan      5         2.0 0.647579       0.037034
dcgan      6         2.0 0.399742       0.052301
 wgan      1         0.5 0.871211      -0.024128
 wgan      2         0.5 0.850688      -0.003806
 wgan      3         1.0 0.715136       0.024497
 wgan      4         2.0 0.682653       0.035077
 wgan      5         2.0 0.637077       0.026819
 wgan      6         2.0 0.416426       0.068846

Full mean+/-std tab